# Compare muscle map to ground truth

In [1]:
# libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from ipywidgets import interact, fixed
from IPython.display import clear_output
import SimpleITK as sitk

In [2]:
!pwd

/c/Projects/dissector/eval_notebooks/myosegmenTUM


In [8]:
# take Muscle Map segmented image, and segmented image
pred_name = "../MuscleMap_segs/HV001_1_FATFRACTION_stack1_dseg.nii"
segment_image = sitk.ReadImage(pred_name)
gt_name = "HV001_1/SegmentationMasks/combined_gt_stack1.mha"
gt_image= sitk.ReadImage(gt_name)# here we need to write match which is FOLDER(same as above fat fraction)/ SegmentationMasks/combined_gt_stack*(from above).mha

In [32]:
results_r_gracilis = []
for file in os.listdir(os.path.join("..","MuscleMap_segs")):
    print(file)
    gt_name = file[:7]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
    print(gt_name)
    segment_image = sitk.ReadImage(os.path.join("..","MuscleMap_segs",file))
    new_segment_array = sitk.GetArrayFromImage(segment_image)
    gt_image= sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)
    # actually lower
    #results = []
    gt = sitk.Cast(gt_image ==  5, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 7152, sitk.sitkUInt8)
    
    gt_arr = sitk.GetArrayFromImage(gt)
    pred_arr = sitk.GetArrayFromImage(pred)
    
    print("GT voxels:", np.sum(gt_arr))
    print("Pred voxels:", np.sum(pred_arr))
    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    
    right_grac_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi  = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive  = dice_filter.GetFalsePositiveError()
    print("Right gracilis lower dice:", right_grac_dice_lower)
    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    
    right_grac_hd_lower = hd_filter.GetHausdorffDistance()
    print("Right gracilis lower Hausdorff distance:", right_grac_hd_lower)
    results_r_gracilis.append({
            "image": gt_name,
            "pred_label": pred_name,
            "R_gracilis_lower_dice:": right_grac_dice_lower,
            "R_gracilis_lower_Hausdorff:": right_grac_hd_lower,
            "R_gracilis_jaccard":right_JaccardCoeffi,
            "R_gracilis_volume_similarity":right_VolumeSimilar,
            "R_gracilis_falseNegative":right_FalseNegative,
            "R_gracilis_falsePostivie":right_FalsePositive,
        })
df_r_gracilis = pd.DataFrame(results_r_gracilis)
df_r_gracilis

HV001_1_FATFRACTION_stack1_dseg.nii.gz
HV001_1/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 15952
Pred voxels: 17119
Right gracilis lower dice: 0.8641407880015723
Right gracilis lower Hausdorff distance: 11.045361017187261
HV001_1_FATFRACTION_stack2_dseg.nii.gz
HV001_1/SegmentationMasks/combined_gt_stack2.mha
GT voxels: 15923
Pred voxels: 16246
Right gracilis lower dice: 0.8808480213870497
Right gracilis lower Hausdorff distance: 16.0
HV001_2_FATFRACTION_stack1_dseg.nii.gz
HV001_2/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 16173
Pred voxels: 17817
Right gracilis lower dice: 0.8444836716681376
Right gracilis lower Hausdorff distance: 14.866068747318506
HV001_2_FATFRACTION_stack2_dseg.nii.gz
HV001_2/SegmentationMasks/combined_gt_stack2.mha
GT voxels: 14708
Pred voxels: 16003
Right gracilis lower dice: 0.8674416332910032
Right gracilis lower Hausdorff distance: 20.0
HV001_3_FATFRACTION_stack1_dseg.nii.gz
HV001_3/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 17379
Pre

,image,pred_label,R_gracilis_lower_dice:,R_gracilis_lower_Hausdorff:,R_gracilis_jaccard,R_gracilis_volume_similarity,R_gracilis_falseNegative,R_gracilis_falsePostivie
0,HV001_1/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.864141,11.045361,0.760782,-0.070575,0.165313,0.000057
1,HV001_1/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.880848,16.000000,0.787067,-0.020081,0.127908,0.000086
2,HV001_2/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.844484,14.866069,0.730828,-0.096734,0.194477,0.000062
3,HV001_2/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.867442,20.000000,0.765913,-0.084335,0.167656,0.000068
4,HV001_3/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.864027,9.055385,0.760606,-0.052282,0.157984,0.000067
5,HV001_3/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.889236,16.031220,0.800562,0.048875,0.088489,0.000106


In [35]:
results_l_gracilis = []
for file in os.listdir(os.path.join("..","MuscleMap_segs")):
    print(file)
    gt_name = file[:7]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
    print(gt_name)
    segment_image = sitk.ReadImage(os.path.join("..","MuscleMap_segs",file))
    new_segment_array = sitk.GetArrayFromImage(segment_image)
    gt_image= sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)
    # actually lower
    #results = []
    gt = sitk.Cast(gt_image ==  1, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 7151, sitk.sitkUInt8)
    
    gt_arr = sitk.GetArrayFromImage(gt)
    pred_arr = sitk.GetArrayFromImage(pred)
    
    print("GT voxels:", np.sum(gt_arr))
    print("Pred voxels:", np.sum(pred_arr))
    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    
    left_grac_dice_lower = dice_filter.GetDiceCoefficient()
    left_JaccardCoeffi  = dice_filter.GetJaccardCoefficient()
    left_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    left_FalseNegative = dice_filter.GetFalseNegativeError()
    left_FalsePositive  = dice_filter.GetFalsePositiveError()
    print("Left gracilis lower dice:", left_grac_dice_lower)
    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    
    left_grac_hd_lower = hd_filter.GetHausdorffDistance()
    print("Left gracilis lower Hausdorff distance:", left_grac_hd_lower)
    results_l_gracilis.append({
            "image": gt_name,
            "pred_label": pred_name,
            "R_gracilis_lower_dice:": left_grac_dice_lower,
            "R_gracilis_lower_Hausdorff:": left_grac_hd_lower,
            "R_gracilis_jaccard":left_JaccardCoeffi,
            "R_gracilis_volume_similarity":right_VolumeSimilar,
            "R_gracilis_falseNegative":left_FalseNegative,
            "R_gracilis_falsePostivie":left_FalsePositive,
        })
df_l_gracilis = pd.DataFrame(results_l_gracilis)
df_l_gracilis

HV001_1_FATFRACTION_stack1_dseg.nii.gz
HV001_1/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 13748
Pred voxels: 14040
Left gracilis lower dice: 0.8478479919389664
Left gracilis lower Hausdorff distance: 8.306623862918075
HV001_1_FATFRACTION_stack2_dseg.nii.gz
HV001_1/SegmentationMasks/combined_gt_stack2.mha
GT voxels: 14946
Pred voxels: 14732
Left gracilis lower dice: 0.8403531235258441
Left gracilis lower Hausdorff distance: 12.206555615733702
HV001_2_FATFRACTION_stack1_dseg.nii.gz
HV001_2/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 13884
Pred voxels: 14792
Left gracilis lower dice: 0.8586971683637885
Left gracilis lower Hausdorff distance: 6.4031242374328485
HV001_2_FATFRACTION_stack2_dseg.nii.gz
HV001_2/SegmentationMasks/combined_gt_stack2.mha
GT voxels: 15171
Pred voxels: 14198
Left gracilis lower dice: 0.8280159351697368
Left gracilis lower Hausdorff distance: 20.199009876724155
HV001_3_FATFRACTION_stack1_dseg.nii.gz
HV001_3/SegmentationMasks/combined_gt_stack1.mha
G

,image,pred_label,R_gracilis_lower_dice:,R_gracilis_lower_Hausdorff:,R_gracilis_jaccard,R_gracilis_volume_similarity,R_gracilis_falseNegative,R_gracilis_falsePostivie
0,HV001_1/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.847848,8.306624,0.735882,0.048875,0.160969,0.000067
1,HV001_1/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.840353,12.206556,0.724663,0.048875,0.153543,0.000122
2,HV001_2/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.858697,6.403124,0.752383,0.048875,0.167658,0.000054
3,HV001_2/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.828016,20.199010,0.706508,0.048875,0.143612,0.000148
4,HV001_3/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.835699,8.124038,0.717769,0.048875,0.181616,0.000073
5,HV001_3/SegmentationMasks/combined_gt_stack2.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.827036,16.248077,0.705083,0.048875,0.185358,0.000106


In [ ]:
#sartorius
results_r_sart = []
for file in os.listdir(os.path.join("..","MuscleMap_segs")):
    print(file)
    gt_name = file[:7]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
    print(gt_name)
    segment_image = sitk.ReadImage(os.path.join("..","MuscleMap_segs",file))
    new_segment_array = sitk.GetArrayFromImage(segment_image)
    gt_image= sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)
    # actually lower
    #results = []
    gt = sitk.Cast(gt_image ==  8, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 7142, sitk.sitkUInt8)
    
    gt_arr = sitk.GetArrayFromImage(gt)
    pred_arr = sitk.GetArrayFromImage(pred)
    
    print("GT voxels:", np.sum(gt_arr))
    print("Pred voxels:", np.sum(pred_arr))
    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    
    right_sart_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi  = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive  = dice_filter.GetFalsePositiveError()
    print("Right sart lower dice:", right_sart_dice_lower)
    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    
    right_sart_hd_lower = hd_filter.GetHausdorffDistance()
    print("Right sart lower Hausdorff distance:", right_sart_hd_lower)
    results_r_sart.append({
            "image": gt_name,
            "pred_label": pred_name,
            "R_sartorius_lower_dice:": right_sart_dice_lower,
            "R_sartorius_lower_Hausdorff:": right_sart_hd_lower,
            "R_sartorius_jaccard":right_JaccardCoeffi,
            "R_sartorius_volume_similarity":right_VolumeSimilar,
            "R_sartorius_falseNegative":right_FalseNegative,
            "R_sartorius_falsePostivie":right_FalsePositive,
        })
df_r_sart = pd.DataFrame(results_r_sart)
df_r_sart

In [36]:
results_l_sart = []
for file in os.listdir(os.path.join("..","MuscleMap_segs")):
    print(file)
    gt_name = file[:7]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
    print(gt_name)
    segment_image = sitk.ReadImage(os.path.join("..","MuscleMap_segs",file))
    new_segment_array = sitk.GetArrayFromImage(segment_image)
    gt_image= sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)
    # actually lower
    #results = []
    gt = sitk.Cast(gt_image ==  4, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 7141, sitk.sitkUInt8)
    
    gt_arr = sitk.GetArrayFromImage(gt)
    pred_arr = sitk.GetArrayFromImage(pred)
    
    print("GT voxels:", np.sum(gt_arr))
    print("Pred voxels:", np.sum(pred_arr))
    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    
    left_sart_dice_lower = dice_filter.GetDiceCoefficient()
    left_JaccardCoeffi  = dice_filter.GetJaccardCoefficient()
    left_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    left_FalseNegative = dice_filter.GetFalseNegativeError()
    left_FalsePositive  = dice_filter.GetFalsePositiveError()
    print("Left sartorius lower dice:", left_sart_dice_lower)
    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    
    left_sart_hd_lower = hd_filter.GetHausdorffDistance()
    print("Left sartoriuss lower Hausdorff distance:", left_sart_hd_lower)
    results_l_sart.append({
            "image": gt_name,
            "pred_label": pred_name,
            "R_sartor_lower_dice:": left_sart_dice_lower,
            "R_sartor_lower_Hausdorff:": left_sart_hd_lower,
            "R_sartor_jaccard":left_JaccardCoeffi,
            "R_sartor_volume_similarity":right_VolumeSimilar,
            "R_sartor_falseNegative":left_FalseNegative,
            "R_sartor_falsePostivie":left_FalsePositive,
        })
df_l_sart = pd.DataFrame(results_l_sart)
df_l_sart

HV001_1_FATFRACTION_stack1_dseg.nii.gz
HV001_1/SegmentationMasks/combined_gt_stack1.mha
GT voxels: 25018
Pred voxels: 27827


NameError: name 'left_sartdice_lower' is not defined

In [9]:
#new_segment_array = sitk.GetArrayFromImage(segment_image)
# # keep only keep labels we care about
# keep_labels = [
#     7101, 7102,
#     7111, 7112,
#     7121, 7122,
#     7131, 7132,
#     7141, 7142,
#     7151, 7152,
#     7161, 7162,
#     7171, 7172,
#     7181, 7182,
#     7191, 7192
# ]

# mask = np.isin(new_segment_array, keep_labels)
# filtered = np.where(mask, new_segment_array, 0)

In [10]:
# filtered_image = sitk.GetImageFromArray(filtered)
# filtered_image.CopyInformation(gt_image)

In [11]:
#segment_image.CopyInformation(gt_image)

In [12]:
# # actually lower
# results = []
# gt = sitk.Cast(gt_image ==  5, sitk.sitkUInt8)
# pred = sitk.Cast(segment_image == 7152, sitk.sitkUInt8)

# gt_arr = sitk.GetArrayFromImage(gt)
# pred_arr = sitk.GetArrayFromImage(pred)

# print("GT voxels:", np.sum(gt_arr))
# print("Pred voxels:", np.sum(pred_arr))
# dice_filter = sitk.LabelOverlapMeasuresImageFilter()
# dice_filter.Execute(gt, pred)

# right_grac_dice_lower = dice_filter.GetDiceCoefficient()
# print("Right gracilis lower dice:", right_grac_dice_lower)
# hd_filter = sitk.HausdorffDistanceImageFilter()
# hd_filter.Execute(gt, pred)

# right_grac_hd_lower = hd_filter.GetHausdorffDistance()
# print("Right gracilis lower Hausdorff distance:", right_grac_hd_lower)
# results.append({
#         "image": gt_name,
#         "pred_label": pred_name,
#         "R_gracilis_lower_dice:": right_grac_dice_lower,
#         "R_gracilis_lower_Hausdorff:": right_grac_hd_lower
#     })
# df = pd.DataFrame(results)
# df

GT voxels: 15952
Pred voxels: 17119
Right gracilis lower dice: 0.8641407880015723
Right gracilis lower Hausdorff distance: 11.045361017187261


,image,pred_label,R_gracilis_lower_dice:,R_gracilis_lower_Hausdorff:
0,HV001_1/SegmentationMasks/combined_gt_stack1.mha,../MuscleMap_segs/HV001_1_FATFRACTION_stack1_d...,0.864141,11.045361
